# 🔍 Understanding CLIP: Connecting Images and Text

**CLIP** (Contrastive Language–Image Pretraining) is a model from OpenAI that learns to associate images with natural language descriptions — trained on hundreds of millions of image-caption pairs from the web.

Unlike traditional vision models that learn from fixed label sets, CLIP encodes **both images and text into a shared embedding space**. This makes it remarkably flexible: you can ask it questions about any image using plain language, without ever fine-tuning it on your specific task.

---

## What we'll cover

| Section | What you'll learn |
|---|---|
| **1. Setup** | Install and load CLIP |
| **2. Zero-shot classification** | Label an image using custom text categories |
| **3. Image–text similarity** | Score how well a description matches an image |
| **4. Image search / ranking** | Find the best-matching image from a set given a query |

---

> **Prerequisites:** Familiarity with Python and basic ML concepts (embeddings, softmax). No prior CLIP experience needed.
>
> **Runtime:** Make sure you're using a **GPU runtime** for faster inference. Go to `Runtime > Change runtime type > T4 GPU`.

---
## Section 1: Setup

We'll use OpenAI's original CLIP implementation via the `clip` package, along with `Pillow` for image loading and `requests` to fetch images from URLs.

In [ ]:
# Install CLIP and dependencies
!pip install git+https://github.com/openai/CLIP.git --quiet
!pip install Pillow requests --quiet

print("✅ Installation complete")

In [ ]:
import clip
import torch
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from io import BytesIO

# Load CLIP model — ViT-B/32 is the standard starting point
# Other options: 'ViT-L/14', 'RN50', 'RN101'
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print(f"✅ CLIP loaded on: {device}")
print(f"   Model: ViT-B/32")
print(f"   Input resolution: {model.visual.input_resolution}px")

In [ ]:
# --- Helper: Load an image from a URL ---
def load_image_from_url(url):
    """Fetch an image from a URL and return a PIL Image."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; CLIP-Tutorial/1.0)"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

def show_image(image, title=None, figsize=(5, 5)):
    """Display a PIL image inline."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=13, pad=10)
    plt.tight_layout()
    plt.show()

print("✅ Helper functions ready")

---
## Section 2: Zero-Shot Image Classification

CLIP can classify an image into any set of categories you define — **no training required**. You simply provide text labels and it picks the best match.

### How it works

1. Encode the image → image embedding vector
2. Encode each label as `"a photo of a {label}"` → text embedding vectors  
3. Compute **cosine similarity** between the image and each label
4. Apply **softmax** to turn similarities into probabilities

The label with the highest probability is CLIP's prediction.

In [ ]:
# ---------------------------------------------------------------
# 🔧 MODIFY THESE — paste any public image URL and your own labels
# ---------------------------------------------------------------
IMAGE_URL = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=600"

LABELS = [
    "cat",
    "dog",
    "bird",
    "fish",
    "rabbit",
]
# ---------------------------------------------------------------

# Load and display the image
image = load_image_from_url(IMAGE_URL)
show_image(image, title="Input image")

In [ ]:
# Encode the image
image_input = preprocess(image).unsqueeze(0).to(device)

# Encode labels — CLIP works better with prompt templates like "a photo of a {label}"
text_prompts = [f"a photo of a {label}" for label in LABELS]
text_tokens = clip.tokenize(text_prompts).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_tokens)

# Normalize and compute cosine similarities
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

# Scale by CLIP's learned temperature parameter
similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
probs = similarity[0].cpu().numpy()

# --- Display results ---
sorted_idx = np.argsort(probs)[::-1]
colors = ["#2563eb" if i == sorted_idx[0] else "#93c5fd" for i in range(len(LABELS))]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh([LABELS[i] for i in sorted_idx], [probs[i] for i in sorted_idx],
               color=[colors[i] for i in sorted_idx])
ax.set_xlabel("Probability", fontsize=11)
ax.set_title("Zero-shot classification results", fontsize=13)
ax.set_xlim(0, 1)
for bar, val in zip(bars, [probs[i] for i in sorted_idx]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.1%}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

print(f"\n🏆 Top prediction: '{LABELS[sorted_idx[0]]}' ({probs[sorted_idx[0]]:.1%})")

### 💡 Things to try

- **Swap in your own image URL** and labels for a dataset you actually work with
- Try **more specific labels**: `"tabby cat"` vs `"siamese cat"` vs `"cat"` — does specificity help?
- Try **confusable categories**: `"wolf"` and `"dog"` on a wolf image — how does CLIP handle near-neighbors?
- Change the prompt template: what happens with `"a drawing of a {label}"` or just `"{label}"`?

---
## Section 3: Image–Text Similarity Scoring

Rather than picking a winner, you can use CLIP to score **how well a specific description matches an image** — useful for retrieval evaluation, content auditing, or understanding what a model "sees" in an image.

In [ ]:
# ---------------------------------------------------------------
# 🔧 MODIFY THESE
# ---------------------------------------------------------------
IMAGE_URL = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=600"

# Multiple descriptions to score against the same image
DESCRIPTIONS = [
    "a cat sitting on a surface",
    "a fluffy orange cat",
    "a dog playing in a park",
    "an animal with pointy ears",
    "a person holding a coffee cup",
    "a close-up photo of a mammal",
]
# ---------------------------------------------------------------

image = load_image_from_url(IMAGE_URL)
show_image(image, title="Input image")

In [ ]:
# Encode image and descriptions
image_input = preprocess(image).unsqueeze(0).to(device)
text_tokens = clip.tokenize(DESCRIPTIONS).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_tokens)

# Normalize
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

# Raw cosine similarity (not softmax — so scores are comparable across queries)
similarities = (image_features @ text_features.T)[0].cpu().numpy()

# Display
sorted_idx = np.argsort(similarities)[::-1]
threshold = 0.25  # rough heuristic: scores > 0.25 are typically meaningful

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ["#16a34a" if similarities[i] >= threshold else "#d1d5db" for i in sorted_idx]
bars = ax.barh(
    [DESCRIPTIONS[i] for i in sorted_idx],
    [similarities[i] for i in sorted_idx],
    color=bar_colors
)
ax.axvline(threshold, color="#dc2626", linestyle="--", linewidth=1, label=f"Threshold ({threshold})")
ax.set_xlabel("Cosine similarity", fontsize=11)
ax.set_title("Image–text similarity scores", fontsize=13)
ax.legend(fontsize=9)
for bar, val in zip(bars, [similarities[i] for i in sorted_idx]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\nScores (higher = more similar):")
for i in sorted_idx:
    match = "✅" if similarities[i] >= threshold else "  "
    print(f"  {match} {similarities[i]:.4f}  —  {DESCRIPTIONS[i]}")

### 💡 Things to try

- Test **negations**: does `"no cats in this image"` score low? (Spoiler: CLIP often struggles with negation)
- Try **very similar descriptions** with small differences: `"a red car"` vs `"a blue car"` on the same car image
- Use this for **dataset auditing**: run it across a set of images to find ones that don't match their captions

> **Note on scores:** Raw cosine similarity values aren't probabilities — they're typically in the range `[0.1, 0.35]` for reasonable matches. The softmax in Section 2 rescales these; here we keep them raw so they're comparable across different text inputs.

---
## Section 4: Image Search / Ranking

Given a **text query** and a **set of images**, CLIP can rank the images by relevance. This is the core of many visual search and retrieval systems.

Provide multiple image URLs below and a query — CLIP will rank them.

In [ ]:
# ---------------------------------------------------------------
# 🔧 MODIFY THESE — add more image URLs to the list
# ---------------------------------------------------------------
TEXT_QUERY = "a dog running outdoors"

IMAGE_URLS = [
    "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",  # cat
    "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=400",  # dog
    "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=400",     # dog 2
]

# Optional short labels for display (or leave blank)
IMAGE_LABELS = ["cat", "dog", "dog"]
# ---------------------------------------------------------------

print(f"Loading {len(IMAGE_URLS)} images...")
images = []
for i, url in enumerate(IMAGE_URLS):
    try:
        img = load_image_from_url(url)
        images.append(img)
        print(f"  ✅ Image {i+1} loaded")
    except Exception as e:
        print(f"  ❌ Image {i+1} failed: {e}")
        images.append(None)

valid_idx = [i for i, img in enumerate(images) if img is not None]
print(f"\n{len(valid_idx)} images ready for ranking.")

In [ ]:
# Encode all valid images
image_inputs = torch.stack([
    preprocess(images[i]) for i in valid_idx
]).to(device)

# Encode the text query
text_token = clip.tokenize([TEXT_QUERY]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image_inputs)
    text_features = model.encode_text(text_token)

# Normalize and score
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

scores = (image_features @ text_features.T).squeeze(1).cpu().numpy()

# Rank by score
ranked_idx = np.argsort(scores)[::-1]

# Display ranked results
n = len(valid_idx)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
if n == 1:
    axes = [axes]

for rank, orig_idx in enumerate(ranked_idx):
    ax = axes[rank]
    img_orig = valid_idx[orig_idx]
    ax.imshow(images[img_orig])
    ax.axis("off")
    label = IMAGE_LABELS[img_orig] if img_orig < len(IMAGE_LABELS) else f"Image {img_orig+1}"
    border_color = "#16a34a" if rank == 0 else "#9ca3af"
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
        spine.set_visible(True)
    medal = ["🥇", "🥈", "🥉"] + ["   "] * 10
    ax.set_title(f"{medal[rank]} Rank {rank+1}\n{label}\nscore: {scores[orig_idx]:.4f}",
                 fontsize=10, pad=6)

fig.suptitle(f'Query: "{TEXT_QUERY}"', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nRanking summary:")
for rank, orig_idx in enumerate(ranked_idx):
    label = IMAGE_LABELS[valid_idx[orig_idx]] if valid_idx[orig_idx] < len(IMAGE_LABELS) else f"Image {valid_idx[orig_idx]+1}"
    print(f"  #{rank+1}  {scores[orig_idx]:.4f}  —  {label}")

### 💡 Things to try

- **Build a small image dataset** from URLs in a domain you study — medical images, satellite imagery, microscopy — and test domain-specific queries
- Try **abstract or relational queries**: `"something sad"`, `"a crowded place"`, `"symmetry"`
- Try **multilingual queries** — CLIP has some multilingual ability depending on the variant
- Compare the ranking with **RN50 vs ViT-L/14** by changing the model name in Section 1

---
## Summary & Next Steps

You've seen three core CLIP capabilities:

| Capability | Key idea |
|---|---|
| **Zero-shot classification** | Use text labels as classifiers — no training data needed |
| **Similarity scoring** | Quantify image–text alignment; useful for retrieval eval |
| **Image ranking** | Sort a set of images by relevance to a natural language query |

### Where to go next

- **[OpenCLIP](https://github.com/mlfoundations/open_clip)** — open-source CLIP variants, including models trained on domain-specific data
- **[SigLIP](https://huggingface.co/docs/transformers/model_doc/siglip)** (Google) — improved loss function, better low-shot performance
- **Fine-tuning CLIP** — adapt to your domain using PEFT/LoRA on domain-specific image-caption pairs
- **CLIP for retrieval pipelines** — combine with a vector database (FAISS, Weaviate) for scalable image search

### Known limitations to keep in mind

- Struggles with **counting, spatial reasoning**, and **negation**
- Performance drops on **highly specialized or technical domains** not well-represented in web data
- Embeddings can reflect **biases** in the training corpus
- Not a generative model — it scores, ranks, and retrieves, but doesn't describe images in free text (for that, see LLaVA, BLIP-2, or GPT-4V)